# Call for Practice

In [2]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import csv

In [8]:
edges_df = pd.read_csv('edges.csv', names=["source", "target"])
df = edges_df.sample(n=50)
df.shape

(50, 2)

In [10]:
G = nx.from_pandas_edgelist(df, source='source', target='target', create_using=nx.DiGraph())

In [14]:
ordre = G.number_of_nodes()
print(f"Ordre du graphe : {ordre} sommets")
taille = G.number_of_edges()
print(f"Taille du graphe : {taille} arêtes\n")

Ordre du graphe : 100 sommets
Taille du graphe : 50 arêtes



In [ ]:
def hits_manual(G, maxiter=100, tol=1e-6):
    nodes = list(G.nodes())
    h = {v: 1.0 for v in nodes}
    a = {v: 0.0 for v in nodes}
    for  in range(max_iter):
        for v in nodes:
            a[v] = sum(h[u] for u in G.neighbors(v))
        for v in nodes:
            h[v] = sum(a[u] for u in G.neighbors(v))
        norm_a = np.sqrt(sum(val  2 for val in a.values()))
        norm_h = np.sqrt(sum(val  2 for val in h.values()))
        for v in nodes:
            a[v] /= norm_a if norm_a != 0 else 1
            h[v] /= norm_h if norm_h != 0 else 1
        if all(abs(a[v] - h[v]) < tol for v in nodes):
            break
    return a, h

authority_dict, hub_dict = hits_manual(G)

In [15]:
pagerank = nx.pagerank(G, alpha=0.85)
hubs, authorities = nx.hits(G)

In [16]:
top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]
top_hubs = sorted(hubs.items(), key=lambda x: x[1], reverse=True)[:10]
top_auth = sorted(authorities.items(), key=lambda x: x[1], reverse=True)[:10]

In [17]:
pos = nx.spring_layout(G, seed=42)

def plot_graph(scores, title, filename):
    plt.figure(figsize=(10, 8))
    nx.draw_networkx_nodes(
        G, pos,
        node_size=[5000 * v for v in scores.values()],
        node_color=list(scores.values()),
        cmap=plt.cm.viridis,
        alpha=0.8
    )
    nx.draw_networkx_edges(G, pos, alpha=0.2)
    nx.draw_networkx_labels(G, pos, font_size=8)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()

plot_graph(pagerank, "PageRank", "pagerank.png")
plot_graph(authorities, "Authority (HITS)", "authority.png")
plot_graph(hubs, "Hub (HITS)", "hub.png")


c:\Users\dylag\AppData\Local\Programs\Python\Python313\Lib\site-packages\matplotlib\collections.py:1008: RuntimeWarning: invalid value encountered in sqrt
  scale = np.sqrt(self._sizes) * dpi / 72.0 * self._factor


In [18]:
pd.DataFrame({
    "node": list(pagerank.keys()),
    "pagerank": list(pagerank.values()),
    "authority": [authorities.get(n, 0) for n in pagerank.keys()],
    "hub": [hubs.get(n, 0) for n in pagerank.keys()]
}).sort_values("pagerank", ascending=False).to_csv("scores.csv", index=False)